In [0]:
# %sql
# -- 1. Aseguramos que la tabla no tenga basura previa
# DROP TABLE IF EXISTS products.bronze_scraped_products;

In [0]:
%sql

CREATE TABLE IF NOT EXISTS workspace.products.bronze_scraped_products (
    scraped_at STRING,
    product_id STRING,
    retailer STRING,
    raw_data STRING,
    ingested_at TIMESTAMP,
    source_file STRING
)
USING DELTA

In [0]:
%sql
-- Rename files with spaces to avoid URI errors
-- Then load data using COPY INTO
COPY INTO workspace.products.bronze_scraped_products
FROM '/Volumes/workspace/products/products_tracker/scraped/'
FILEFORMAT = PARQUET
PATTERN = 'year=*/month=*/day=*/*.parquet'
FORMAT_OPTIONS ('mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
%sql
SELECT
    product_id,
    retailer,
    get_json_object(raw_data, '$.list_price') as list_price,
    get_json_object(raw_data, '$.cash_price') as cash_price,
    scraped_at
FROM
    products.bronze_scraped_products
WHERE
    scraped_at = current_date()

In [0]:
%sql
WITH last_two_price_updates AS (
  SELECT
    product_id,
    retailer,
    get_json_object(raw_data, '$.list_price') as list_price,
    get_json_object(raw_data, '$.cash_price') as cash_price,
    scraped_at,
    ROW_NUMBER() OVER (PARTITION BY product_id, retailer ORDER BY scraped_at DESC) as rn
  FROM
    products.bronze_scraped_products
)
SELECT
  product_id,
  retailer,
  list_price,
  cash_price,
  scraped_at
FROM last_two_price_updates
WHERE rn <= 2
ORDER BY RIGHT(product_id, 3), retailer, scraped_at DESC;

In [0]:
%sql
SELECT 
*
FROM 
parquet.`/Volumes/workspace/products/products_tracker/errors/`